In [3]:
import pandas as pd
import numpy as np
from statsmodels.tsa.ar_model import AutoReg
import warnings

warnings.filterwarnings("ignore")


In [42]:
x="/Users/harold/DataAnalyctisandScience/Cointegration_Johansen_test/dataHousingCPI/UEMOA"

In [43]:
# 1. Charger le fichier CSV

df = pd.read_excel(x+".xlsx") 

# Vérifier les valeurs manquantes
print("Valeurs manquantes par colonne :")
print(df.isna().sum())
print("\nAperçu des données :")
print(df.head())

Valeurs manquantes par colonne :
Months                        0
House CPI                     0
EUR/USD                       0
Crude oil, average ($/bbl)    2
Crude oil, Brent ($/bbl)      2
Crude oil, WTI ($/bbl)        2
dtype: int64

Aperçu des données :
      Months  House CPI   EUR/USD  Crude oil, average ($/bbl)  \
0 2000-01-01       0.96  1.013695                       25.31   
1 2000-02-01       1.01  0.983424                       27.22   
2 2000-03-01       1.09  0.964343                       27.49   
3 2000-04-01       1.27  0.946950                       23.47   
4 2000-05-01       1.55  0.905968                       27.19   

   Crude oil, Brent ($/bbl)  Crude oil, WTI ($/bbl)  
0                     25.38                   27.27  
1                     27.70                   29.28  
2                     27.47                   29.92  
3                     22.54                   25.84  
4                     27.34                   28.83  


In [44]:
# 2. Fonction pour choisir le meilleur nombre de lags

def meilleur_lag(serie, max_lags=10):
    serie = serie.dropna()
    if len(serie) < 5:  # Trop peu de points pour un modèle AR
        return 1
    aic_values = {}
    for lag in range(1, min(max_lags, len(serie)//2)):
        try:
            model = AutoReg(serie, lags=lag, old_names=False)
            result = model.fit()
            aic_values[lag] = result.aic
        except:
            continue
    return min(aic_values, key=aic_values.get) if aic_values else 1


# 3. Fonction pour remplir les NaN avec AR

def Interpol_AR_backcaster(serie):
    if serie.isna().sum() == len(serie):
        return serie
    
    # Nombre de valeurs manquantes au début
    backcast_steps = 0
    for val in serie:
        if pd.isna(val):
            backcast_steps += 1
        else:
            break
    
    # Interpolation initiale
    serie_init = serie.interpolate(method="linear", limit_direction="both")
    
    # Déterminer le meilleur lag
    lags = meilleur_lag(serie_init)
    
    # Entraîner AR
    model = AutoReg(serie_init, lags=lags, old_names=False)
    model_fit = model.fit()
    
    # Prédire toute la série existante
    prediction = model_fit.predict(start=0, end=len(serie_init)-1)
    
    # Remplacer les NaN internes
    serie_finale = serie.copy()
    serie_finale[serie_finale.isna()] = prediction[serie_finale.isna()]
    
    # 🔄 Backcasting (uniquement si on a des NaN au début)
    backcast_vals = []
    history = serie_finale.dropna().tolist()
    
    for i in range(backcast_steps):
        coeffs = model_fit.params
        lags_values = history[:lags]  # premiers points connus
        back_val = np.mean(lags_values) if lags_values else history[0]
        backcast_vals.append(back_val)
    
    # Ajouter les valeurs estimées au début
    backcast_vals = backcast_vals[::-1]  # ordre chronologique
    serie_complete = pd.Series(backcast_vals + serie_finale.tolist())
    
    return serie_complete


In [45]:
# 4. Appliquer la fonction à toutes les colonnes numériques

df_complet = df.copy()

for col in df.columns:
    if df[col].dtype in [np.float64, np.int64]:  
        print(f"🔄 Traitement de la colonne : {col}")
        
        # Tant qu'il reste des valeurs manquantes, on continue
        iteration = 0
        while df_complet[col].isna().sum() > 0:
            iteration += 1
            print(f"   ➝ Itération {iteration} : {df_complet[col].isna().sum()} valeurs manquantes")
            
            df_complet[col] = Interpol_AR_backcaster(df_complet[col])
            
            # Sécurité : éviter boucle infinie si jamais ça bloque
            if iteration > 10:
                print(f"⚠️  Trop d'itérations pour la colonne {col}, arrêt forcé.")
                break
            


🔄 Traitement de la colonne : House CPI
🔄 Traitement de la colonne : EUR/USD
🔄 Traitement de la colonne : Crude oil, average ($/bbl)
   ➝ Itération 1 : 2 valeurs manquantes
🔄 Traitement de la colonne : Crude oil, Brent ($/bbl)
   ➝ Itération 1 : 2 valeurs manquantes
🔄 Traitement de la colonne : Crude oil, WTI ($/bbl)
   ➝ Itération 1 : 2 valeurs manquantes


In [46]:

# Vérifier si toutes les NaN ont disparu
print("\nValeurs manquantes après traitement :")
print(df_complet.isna().sum())

# 5. Sauvegarder le fichier complété
df_complet.to_excel(x+"_completes.xlsx", index=False)
print("\n✅ Fichier Excel complété enregistré :", x+"_completes.xlsx")



Valeurs manquantes après traitement :
Months                        0
House CPI                     0
EUR/USD                       0
Crude oil, average ($/bbl)    0
Crude oil, Brent ($/bbl)      0
Crude oil, WTI ($/bbl)        0
dtype: int64

✅ Fichier Excel complété enregistré : /Users/harold/DataAnalyctisandScience/Cointegration_Johansen_test/dataHousingCPI/UEMOA_completes.xlsx
